# 3b: Hybrid Two-Stage Pipeline (MCQ Benchmark)
This notebook is built directly upon the original `3_vllm_evaluation.ipynb`.
It evaluates all 6 tasks (DTR, BBR, BDC, DRE, DSR, ORR) and generates valid A/B/C/D answers.

**Hybrid Injections:**
1. **BDC Task:** Mathematically counts U-Net bounding boxes and maps to the closest numerical option (e.g., A/B/C/D). Bypasses VLM.
2. **Other Tasks:** Crops bounding boxes, stitches them into a single high-res collage, and feeds it to the VLM to beat the 512-token cap.

# vLLM Evaluation
This notebook handles the vLLM evaluation step.

In [1]:
# Cell 1b: vLLM Install
# Do NOT pin torch — Kaggle's system torch 2.11 is what all other packages expect.
# vLLM 0.6.3 is too old for Qwen2.5-VL's M-RoPE. We need 0.7.2+ for proper support.
!pip install -q vllm qwen-vl-utils kaggle
!pip install -q nvidia-cuda-nvrtc-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 68.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 53.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 30.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 33.5 MB/s eta 0:00:

In [6]:
# Cell 1b: vLLM & U-Net Install
!pip install -q segmentation-models-pytorch opencv-python-headless


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 1.3 MB/s eta 0:00:00a 0:00:01


## Step C: Run vLLM Evaluation with Fallbacks
Intelligently falls back for both the model and the evaluation script if they aren't already downloaded.

In [7]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# import os

# try:
#     # Fetch the token securely from Kaggle Secrets
#     user_secrets = UserSecretsClient()
#     hf_token = user_secrets.get_secret("HF_TOKEN")
    
#     # Log in to Hugging Face Hub
#     login(token=hf_token)
    
#     # Also set it as an environment variable (vLLM automatically looks for this)
#     os.environ["HF_TOKEN"] = hf_token 
    
#     print("✅ Successfully logged into Hugging Face!")
# except Exception as e:
#     print("❌ Could not find HF_TOKEN in Kaggle Secrets. Please add it via Add-ons -> Secrets.")
#     print("Error:", e)

In [8]:
# ── Restore Previous Results ──
import shutil
import os

# Upload your saved "DisasterM3 Evaluation Results" folder to your Kaggle Dataset,
# then mount it as an input dataset. Adjust the path below accordingly:
prev_results = "/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-evaluation-results"

if os.path.exists(prev_results):
    # Copy each subset's results into /tmp/results/ so the script resumes
    for subset in os.listdir(prev_results):
        src = os.path.join(prev_results, subset)
        dst = os.path.join("/tmp/results", subset)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print("✓ Restored previous results for resume!")
else:
    print("No previous results found, starting fresh.")


No previous results found, starting fresh.


In [9]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("✅ Successfully logged into Hugging Face!")
except Exception as e:
    print("❌ Could not find HF_TOKEN in Kaggle Secrets.")
    print("Error:", e)

import json
import shutil

model_path = "/tmp/disasterm3_merged"
if not os.path.exists(model_path):
    print("Local merged model not found in /tmp. Falling back to Hugging Face Hub...")
    model_path = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"
else:
    print(f"Using local merged model at {model_path}")

repo_path = "/tmp/DisasterM3_Repo"
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
!git clone https://github.com/Junjue-Wang/DisasterM3.git {repo_path}
print("✓ Fresh clone of DisasterM3 repo")

# ── Patch run_vllm.py ──
vllm_script = f"{repo_path}/pyscripts/run_vllm.py"
with open(vllm_script, "r") as f:
    lines = f.readlines()

new_lines = []
skip_until_llm = False
for line in lines:
    # 1. Fix the LLM initialization (T4x2 Tensor Parallelism)
    if "EngineArgs(" in line and "engine_args" in line:
        skip_until_llm = True
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}vlm_model = LLM(\n")
        new_lines.append(f"{indent}    model=args.model_id,\n")
        new_lines.append(f"{indent}    limit_mm_per_prompt={{'image': 2}},\n")
        new_lines.append(f"{indent}    gpu_memory_utilization=0.92,\n")
        new_lines.append(f"{indent}    max_model_len=4096,\n")
        new_lines.append(f"{indent}    enforce_eager=True,\n")
        new_lines.append(f"{indent}    trust_remote_code=True,\n")
        new_lines.append(f"{indent}    tensor_parallel_size=2,\n")
        new_lines.append(f"{indent})\n")
        continue
    if skip_until_llm:
        if "LLM(**engine_args)" in line:
            skip_until_llm = False
            continue
        else:
            continue
            
    # 2. Fix the SamplingParams setattr crash
    if "setattr(sampling_params" in line:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}pass  # [PATCHED] Bypass read-only setattr crash\n")
        continue

    # 3. Fix the original authors' relational reasoning bug (Missing "images" folder)
    if 'image_path = join(f"{PROJECT_ROOT}/data", data_dict["image_path"]' in line:
        line = line.replace('f"{PROJECT_ROOT}/data", data_dict', 'f"{PROJECT_ROOT}/data", "images", data_dict')
        
    # 4. Fix the original authors' relational reasoning bug (options_str typo)
    if 'options_str=data_dict["option_str"])' in line:
        line = line.replace('option_str', 'options_str')

    # 5. Globally fix Windows backslashes in all image paths at runtime
    line = line.replace('data_dict["pre_image_path"]', 'data_dict["pre_image_path"].replace("\\\\", "/")')
    line = line.replace('data_dict["post_image_path"]', 'data_dict["post_image_path"].replace("\\\\", "/")')
    line = line.replace('data_dict["image_path"]', 'data_dict["image_path"].replace("\\\\", "/")')

    # 6. Catch corrupted images and gracefully skip them!
    if 'images = [Image.open(image_path).convert("RGB") for image_path in images]' in line:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f'{indent}try:\n')
        new_lines.append(f'{indent}    images = [Image.open(image_path).convert("RGB") for image_path in images]\n')
        new_lines.append(f'{indent}except Exception as e:\n')
        new_lines.append(f'{indent}    print(f"\\nSkipping corrupted image batch: {{e}}")\n')
        new_lines.append(f'{indent}    continue\n')
        continue
    # 7. Remove the broken request_id assertion that crashes when items are skipped
    if "assert int(output.request_id) %" in line:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}pass  # [PATCHED] Removed strict batch size assertion\n")
        continue

    # 8. STOP KAGGLE CPU RAM LEAKS! Force garbage collection after every item
    if "progress_bar.update(1)" in line:
        new_lines.append(line)
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}import gc; gc.collect()\n")
        continue

    new_lines.append(line)
with open(vllm_script, "w") as f:
    f.writelines(new_lines)
print("✓ Patched run_vllm.py (FP16 + SamplingParams + Relational Reasoning + Path + Corrupted Image fixes)")


# ── Data Restructuring ──
bench_path = "/kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2"
data_dir = "/tmp/data"
os.makedirs(f"{data_dir}/images/test_images", exist_ok=True)
os.makedirs(f"{data_dir}/images/masks", exist_ok=True)

!cp -rsn {bench_path}/test_images/* {data_dir}/images/test_images/ 2>/dev/null || true
!cp -rsn {bench_path}/masks/* {data_dir}/images/masks/ 2>/dev/null || true

bench_json_path = f"{bench_path}/benchmark_release.json"
print(f"Splitting {bench_json_path} into evaluation subsets...")
with open(bench_json_path, 'r', encoding='utf-8') as f:
    raw_json = f.read()

# [NEW FIX] Replace all Windows backslashes with Linux forward slashes!
raw_json = raw_json.replace("\\\\", "/")
bench_data = json.loads(raw_json)

task_map = {
    "Disaster Bearing Bodies Recognition": "bearing_body",
    "Building Damage Counting": "building_damage_counting",
    "Disaster Type Recognition": "disaster_type",
    "Road Damage Counting": "road_damage_counting",
    "Disaster Scene Recognition": "landuse",
    "Relational Reasoning": "relational_reasoning_qa"
}

subsets_data = {v: [] for v in task_map.values()}
for entry in bench_data:
    task = entry.get("task")
    if task in task_map:
        subsets_data[task_map[task]].append(entry)

for subset_name, data in subsets_data.items():
    with open(f"{data_dir}/{subset_name}.json", 'w', encoding='utf-8') as f:
        json.dump(data, f)



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Successfully logged into Hugging Face!
Local merged model not found in /tmp. Falling back to Hugging Face Hub...
Cloning into '/tmp/DisasterM3_Repo'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 82 (delta 24), reused 24 (delta 24), pack-reused 56 (from 1)
Receiving objects: 100% (82/82), 22.22 KiB | 689.00 KiB/s, done.
Resolving deltas: 100% (26/26), done.
✓ Fresh clone of DisasterM3 repo
✓ Patched run_vllm.py (FP16 + SamplingParams + Relational Reasoning + Path + Corrupted Image fixes)
Splitting /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/benchmark_release.json into evaluation subsets...


In [ ]:
# ── Stage 1 & 1.5: Hybrid U-Net Cropping & BDC Solving ──
import os
import cv2
import json
import torch
import math
import numpy as np
from PIL import Image
from tqdm import tqdm
import segmentation_models_pytorch as smp
from huggingface_hub import hf_hub_download

print("🚀 Starting Stage 1: U-Net Spatial Extraction")

# 1. Load U-Net from Hugging Face
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
unet_model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=3,
    classes=4,
)

print("Downloading best_model.pth from Hugging Face...")
weights_path = hf_hub_download(
    repo_id="AbrarAlam/disasterm3-unet-checkpoints-2", 
    filename="best_model.pth"
)
unet_model.load_state_dict(torch.load(weights_path, map_location=device))
unet_model.to(device)
unet_model.eval()

# 2. Iterate through Test Images
data_dir = "/tmp/data"
img_dir = f"{data_dir}/images/test_images"
images = [f for f in os.listdir(img_dir) if f.endswith(('.png', '.jpg'))]

print("Scanning images and generating semantic collages...")
for img_name in tqdm(images, desc="U-Net Processing"):
    img_path = os.path.join(img_dir, img_name)
    
    # Read Image
    original_img = cv2.imread(img_path)
    if original_img is None:
        continue
        
    img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    # Preprocess for U-Net (512x512)
    img_resized = cv2.resize(img_rgb, (512, 512))
    img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    # Infer
    with torch.no_grad():
        logits = unet_model(img_tensor)
        preds = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
    
    # Find Damaged/Destroyed pixels (Classes 2 & 3)
    damage_mask = np.where((preds == 2) | (preds == 3), 255, 0).astype(np.uint8)
    
    # Scale mask back to original size (1024x1024)
    damage_mask_full = cv2.resize(damage_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    
    # Extract Bounding Boxes
    contours, _ = cv2.findContours(damage_mask_full, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Filter tiny noise contours (area < 100)
    valid_boxes = [cv2.boundingRect(c) for c in contours if cv2.contourArea(c) > 100]

    # [NEW] Save exact U-Net count for BDC to bypass VLM entirely!
    os.makedirs("/tmp/results/building_damage_counting/AbrarAlam--disasterm3-qwen2.5vl7b-mergedFP/", exist_ok=True)
    bdc_output_path = "/tmp/results/building_damage_counting/AbrarAlam--disasterm3-qwen2.5vl7b-mergedFP/predictions.jsonl"
    bdc_count = len(valid_boxes)
    with open(bdc_output_path, "a") as bdc_f:
        json.dump({"image": img_name, "unet_damage_count": bdc_count}, bdc_f)
        bdc_f.write("\n")
    
    # Create Grid Collage of top 9 damaged regions
    # Sort boxes by area (descending)
    valid_boxes.sort(key=lambda b: b[2] * b[3], reverse=True)
    top_boxes = valid_boxes[:9]
    
    if len(top_boxes) > 0:
        # We will create a 3x3 grid (1024x1024 total, each cell ~341x341)
        grid_size = math.ceil(math.sqrt(len(top_boxes)))
        cell_size = 1024 // max(grid_size, 1)
        
        collage = np.zeros((1024, 1024, 3), dtype=np.uint8)
        
        for idx, (x, y, bw, bh) in enumerate(top_boxes):
            # Crop region with 10% padding
            pad_x = int(bw * 0.1)
            pad_y = int(bh * 0.1)
            x1 = max(0, x - pad_x)
            y1 = max(0, y - pad_y)
            x2 = min(w, x + bw + pad_x)
            y2 = min(h, y + bh + pad_y)
            
            crop = original_img[y1:y2, x1:x2]
            crop_resized = cv2.resize(crop, (cell_size, cell_size))
            
            # Place in grid
            row = idx // grid_size
            col = idx % grid_size
            start_y, start_x = row * cell_size, col * cell_size
            
            collage[start_y:start_y+cell_size, start_x:start_x+cell_size] = crop_resized
            
        # OVERWRITE original image with Collage so Qwen natively loads it!
        cv2.imwrite(img_path, collage)

print("✓ Finished generating high-resolution damage collages.")

# 3. CRITICAL: Unload U-Net to free 16GB VRAM for Qwen!
print("Unloading U-Net from VRAM...")
del unet_model
import gc
gc.collect()
torch.cuda.empty_cache()
print("✓ VRAM cleared. Ready for Stage 2 (Qwen VLM).")


🚀 Starting Stage 1: U-Net Spatial Extraction


best_model.pth:   0%|          | 0.00/97.9M [00:00<?, ?B/s]

Scanning images and generating semantic collages...


U-Net Processing:   1%|          | 52/6260 [00:03<07:15, 14.26it/s]

In [ ]:
# ── Run Evaluation ──

# --- SPLIT 1: Run these first (will take ~6-7 hours) ---
subsets = ["bearing_body", "disaster_type"] 
# Note: "building_damage_counting" is deliberately skipped here because Stage 1.5 already solved it!

# --- SPLIT 2: Next session, comment out Split 1 and uncomment Split 2 ---
# subsets = ["road_damage_counting", "landuse", "relational_reasoning_qa"]

hf_token_val = os.environ.get("HF_TOKEN", "")
safe_model_name = model_path.replace("/", "--")

for subset in subsets:
    print(f"\n{'='*50}\nEvaluating {subset}...\n{'='*50}")

    save_dir = f"/tmp/results/{subset}/{safe_model_name}"
    os.makedirs(save_dir, exist_ok=True)
    with open(f"{save_dir}/finished.jsonl", "a") as f:
        pass

    !HF_TOKEN={hf_token_val} PYTHONPATH={repo_path} python {repo_path}/pyscripts/run_vllm.py --model_id {model_path} --subset {subset} --image_size 512


In [ ]:
# ── Save Results (Working Directory + Kaggle Dataset) ──
import os
import json
import shutil
from kaggle_secrets import UserSecretsClient

source_dir = "/tmp/results"

# =================================================================
# 1. Zip to Kaggle Working Directory
# =================================================================
dest_zip = "/kaggle/working/disasterm3_eval_results"
print(f"Compressing {source_dir} to {dest_zip}.zip ...")
shutil.make_archive(dest_zip, 'zip', source_dir)

size_mb = os.path.getsize(f"{dest_zip}.zip") / (1024 * 1024)
print(f"✓ Results successfully archived! Size: {size_mb:.2f} MB")

# =================================================================
# 2. Upload directly to Kaggle Datasets
# =================================================================
# Authenticate with Kaggle API
try:
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
except Exception as e:
    print("\n❌ Could not find KAGGLE_USERNAME or KAGGLE_KEY in Kaggle Secrets! Please add them to push to datasets.")
    raise e

# Prepare the dataset metadata
dataset_slug = "disasterm3-evaluation-results"  # Lowercase alphanumeric and hyphens only!
username = os.environ["KAGGLE_USERNAME"]

metadata = {
  "title": "DisasterM3 Evaluation Results",
  "id": f"{username}/{dataset_slug}",
  "licenses": [{"name": "CC0-1.0"}]
}

with open(f"{source_dir}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

print(f"\nUploading {source_dir} to Kaggle Datasets as '{dataset_slug}'...")

# Attempt to create the dataset. If it already exists, push a new version instead!
!kaggle datasets create -p {source_dir} --dir-mode zip 2>/dev/null || kaggle datasets version -p {source_dir} -m "Evaluation Update" --dir-mode zip

print("✓ Upload complete! Your results are safely preserved in both /kaggle/working and Kaggle Datasets.")
